In [ ]:
!pip install -U langgraph langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.4
    Uninstalling langgraph-1.1.4:
      Successfully uninstalled langgraph-1.1.4


In [ ]:
import os
import json
from typing import Annotated, TypedDict
from google.colab import userdata # For your API Key

# LangChain / Groq Imports
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# LangGraph Core Imports
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# Initialize your LLM (using the stable 2026 model)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=userdata.get('RealEst')
)

In [ ]:
#defining the state
class SupervisorState(TypedDict):
  messages : Annotated[list , add_messages]
  draft_email : str
  is_approved : bool


#Memory :

memory = MemorySaver()


In [ ]:
#email drafter node
def email_drafter(state : SupervisorState):
  msg = state['messages'][-1].content
  fullMsg = f"You are a professional email writer . Write a professional inquiry based on the following message : {msg} "
  res = llm.invoke(fullMsg)
  return {"draft_email" : res.content}

In [ ]:
# email approving node
def email_sender(state : SupervisorState):
  if state["is_approved"]:
    print(f"Email sent successfully! \n here is the email : \n {state['draft_email']}")
  else:
    print("Email not approved ")
  return {}


In [ ]:
builder = StateGraph(SupervisorState)
builder.add_node("draft",email_drafter)
builder.add_node("sender", email_sender)
builder.add_edge(START, "draft")
builder.add_edge("draft", "sender")
builder.add_edge("draft","sender")
builder.add_edge("sender", END)




In [ ]:
app = builder.compile(checkpointer=memory, interrupt_before=["sender"])

In [ ]:
# 1. Setup the session
config = {"configurable": {"thread_id": "final_project"}}

# 2. Provide the user request
initial_input = {
    "messages": [HumanMessage(content="Write an email to the landlord of the apartment in Shoreditch. Ask if the deposit is refundable.")],
    "is_approved": False # Important to start as False!
}

# 3. Stream the execution
for event in app.stream(initial_input, config):
    print(event)

{'draft': {'draft_email': "Subject: Inquiry Regarding Deposit Refund for Shoreditch Apartment\n\nDear [Landlord's Name],\n\nI hope this email finds you well. I am writing to inquire about the deposit I paid when I moved into the apartment in Shoreditch. As my tenancy is coming to an end, I would like to know if the deposit is refundable and what conditions need to be met in order to receive a full or partial refund.\n\nCould you please provide me with information on the following:\n\n* The deposit amount that was paid and the terms under which it was taken\n* Any deductions that may be made from the deposit, such as damages or unpaid rent\n* The process for requesting a deposit refund and the timeline for receiving the refund\n* Any additional documentation or inspections that may be required to facilitate the refund\n\nI would appreciate it if you could provide me with a clear understanding of the deposit refund process and any relevant details that I need to be aware of. If there are

In [ ]:
# 1. Peek at the draft the AI wrote
current_state = app.get_state(config)
print("--- DRAFT CREATED ---")
print(current_state.values["draft_email"])

# 2. If you like it, flip the switch
# This updates the 'save-game' in the checkpointer
app.update_state(config, {"is_approved": True})

# 3. Resume! Passing 'None' tells LangGraph to pick up from the breakpoint
print("\n--- RESUMING GRAPH ---")
for event in app.stream(None, config):
    print(event)

--- DRAFT CREATED ---
Subject: Inquiry Regarding Deposit Refund for Shoreditch Apartment

Dear [Landlord's Name],

I hope this email finds you well. I am writing to inquire about the deposit I paid when I moved into the apartment in Shoreditch. As my tenancy is coming to an end, I would like to know if the deposit is refundable and what conditions need to be met in order to receive a full or partial refund.

Could you please provide me with information on the following:

* The deposit amount that was paid and the terms under which it was taken
* Any deductions that may be made from the deposit, such as damages or unpaid rent
* The process for requesting a deposit refund and the timeline for receiving the refund
* Any additional documentation or inspections that may be required to facilitate the refund

I would appreciate it if you could provide me with a clear understanding of the deposit refund process and any relevant details that I need to be aware of. If there are any specific requ